# 02 感知机与多层感知机

上一节我们把神经元理解成：输入经过加权求和，再通过激活函数得到输出。

这一节继续往前走：当一个神经元用于分类时，它就是最早期的感知机；当多个神经元堆成隐藏层时，就得到多层感知机，也就是 MLP。

## 1. 学习目标

学完这一节，需要能回答下面几个问题：

1. 感知机和上一节的神经元有什么关系？
2. 为什么感知机可以看成一个线性分类器？
3. 什么是线性可分？
4. 为什么单层感知机无法解决 XOR 问题？
5. MLP 为什么比单层感知机表达能力更强？

## 2. 从神经元到感知机

感知机可以理解成一个用于二分类的神经元。它先计算线性部分：

$$
z = \mathbf{w}^{T}\mathbf{x} + b
$$

然后通过阶跃函数得到类别预测：

$$
\hat{y} = \operatorname{step}(z)
$$

其中阶跃函数可以写成：

$$
\operatorname{step}(z)=
\begin{cases}
1, & z > 0 \\
0, & z \le 0
\end{cases}
$$

所以感知机的核心动作就是：把输入空间切成两边，一边预测为 $1$，另一边预测为 $0$。

## 3. 决策边界

感知机的分界线由下面这个式子决定：

$$
\mathbf{w}^{T}\mathbf{x} + b = 0
$$

如果输入只有两个特征 $x_1$ 和 $x_2$，那么：

$$
w_1x_1 + w_2x_2 + b = 0
$$

当 $w_2 \ne 0$ 时，可以整理成直线形式：

$$
x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}
$$

这条直线就是二维输入空间中的决策边界。高维情况下，它对应的是超平面。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def plot_logic_points(X, y, title):
    plt.figure(figsize=(4.6, 4.2))
    colors = np.where(y == 1, '#DC2626', '#2563EB')
    plt.scatter(X[:, 0], X[:, 1], c=colors, s=120, edgecolor='white', linewidth=1.5)
    for i, (x1, x2) in enumerate(X):
        plt.text(x1 + 0.04, x2 + 0.04, f'y={y[i]}', fontsize=12)
    plt.xlim(-0.3, 1.3)
    plt.ylim(-0.3, 1.3)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.show()

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])

plot_logic_points(X, y_and, 'AND 逻辑')
plot_logic_points(X, y_or, 'OR 逻辑')
plot_logic_points(X, y_xor, 'XOR 逻辑')


## 4. 线性可分

如果可以用一条直线把两类样本分开，这个二分类问题就叫线性可分。

AND 问题是线性可分的。可以构造一个感知机：

$$
z = x_1 + x_2 - 1.5
$$

$$
\hat{y} = \operatorname{step}(z)
$$

当且仅当 $x_1=1$ 且 $x_2=1$ 时，$z>0$，预测结果为 $1$。

In [ ]:
def step(z):
    return (z > 0).astype(int)

w = np.array([1.0, 1.0])
b = -1.5

z = X @ w + b
y_hat = step(z)

print('z:', z)
print('预测:', y_hat)
print('真实:', y_and)

plt.figure(figsize=(4.8, 4.3))
colors = np.where(y_and == 1, '#DC2626', '#2563EB')
plt.scatter(X[:, 0], X[:, 1], c=colors, s=120, edgecolor='white', linewidth=1.5)

x1_line = np.linspace(-0.2, 1.2, 100)
x2_line = 1.5 - x1_line
plt.plot(x1_line, x2_line, color='#111827', linewidth=2, label='$x_1+x_2-1.5=0$')

plt.xlim(-0.3, 1.3)
plt.ylim(-0.3, 1.3)
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('AND 的决策边界')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 5. 为什么 XOR 问题更难

XOR 的规则是：两个输入不同，输出为 $1$；两个输入相同，输出为 $0$。

$$
\begin{array}{c|c|c}
x_1 & x_2 & y \\
\hline
0 & 0 & 0 \\
0 & 1 & 1 \\
1 & 0 & 1 \\
1 & 1 & 0
\end{array}
$$

从图上看，两个正样本在对角线上，两个负样本在另一条对角线上。你无法用一条直线把它们完全分开。

这就是单层感知机的局限：它只能表达线性决策边界。

## 6. 多层感知机 MLP

多层感知机通过加入隐藏层，让模型可以组合多个简单边界，形成更复杂的非线性决策边界。

一个两层 MLP 可以写成：

$$
\mathbf{h} = \sigma(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1)
$$

$$
\hat{y} = g(\mathbf{W}_2\mathbf{h} + b_2)
$$

其中：

- $\mathbf{x}$ 是输入。
- $\mathbf{h}$ 是隐藏层输出。
- $\sigma$ 是隐藏层激活函数。
- $g$ 是输出层激活函数。
- $\mathbf{W}_1, \mathbf{b}_1, \mathbf{W}_2, b_2$ 都是需要学习的参数。

In [ ]:
from matplotlib.patches import Circle, FancyArrowPatch

fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.set_xlim(0, 9)
ax.set_ylim(0, 5)
ax.axis('off')

def node(x, y, text, color):
    circle = Circle((x, y), 0.34, facecolor=color, edgecolor='#1F2937', linewidth=1.5)
    ax.add_patch(circle)
    ax.text(x, y, text, ha='center', va='center', fontsize=13)

def arrow(start, end):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='->', mutation_scale=13, linewidth=1.4, color='#475569'))

input_nodes = [(1.2, 3.2, r'$x_1$'), (1.2, 1.8, r'$x_2$')]
hidden_nodes = [(4.4, 3.4, r'$h_1$'), (4.4, 2.5, r'$h_2$'), (4.4, 1.6, r'$h_3$')]
output_nodes = [(7.5, 2.5, r'$\hat{y}$')]

for x_pos, y_pos, label in input_nodes:
    node(x_pos, y_pos, label, '#EFF6FF')
for x_pos, y_pos, label in hidden_nodes:
    node(x_pos, y_pos, label, '#F0FDF4')
for x_pos, y_pos, label in output_nodes:
    node(x_pos, y_pos, label, '#FEF2F2')

for sx, sy, _ in input_nodes:
    for tx, ty, _ in hidden_nodes:
        arrow((sx + 0.36, sy), (tx - 0.36, ty))

for sx, sy, _ in hidden_nodes:
    arrow((sx + 0.36, sy), (7.5 - 0.36, 2.5))

ax.text(1.2, 4.25, '输入层', ha='center', fontsize=13, weight='bold')
ax.text(4.4, 4.25, '隐藏层', ha='center', fontsize=13, weight='bold')
ax.text(7.5, 4.25, '输出层', ha='center', fontsize=13, weight='bold')
ax.text(2.75, 0.65, r'$\mathbf{W}_1,\mathbf{b}_1$', ha='center', fontsize=13)
ax.text(5.95, 0.65, r'$\mathbf{W}_2,b_2$', ha='center', fontsize=13)

plt.title('一个简单的多层感知机 MLP', fontsize=16, pad=10)
plt.show()


## 7. 批量数据的矩阵形式

实际训练时，我们通常不是一次只输入一个样本，而是一次输入一批样本。

假设批量输入是：

$$
\mathbf{X} \in \mathbb{R}^{m \times n}
$$

其中 $m$ 表示样本数量，$n$ 表示每个样本的特征数量。

如果隐藏层有 $k$ 个神经元，那么：

$$
\mathbf{W}_1 \in \mathbb{R}^{n \times k}, \quad \mathbf{b}_1 \in \mathbb{R}^{k}
$$

隐藏层输出为：

$$
\mathbf{H} = \sigma(\mathbf{X}\mathbf{W}_1 + \mathbf{b}_1)
$$

此时：

$$
\mathbf{H} \in \mathbb{R}^{m \times k}
$$

理解张量形状，是后面写 PyTorch 网络时非常重要的基本功。

## 8. 损失函数与参数更新

感知机最初有自己的参数更新规则。进入现代神经网络后，我们通常统一使用损失函数和梯度下降来训练。

二分类常用的二元交叉熵损失是：

$$
\mathcal{L}(\hat{y}, y) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log\hat{y}^{(i)} + (1-y^{(i)})\log(1-\hat{y}^{(i)})\right]
$$

梯度下降的更新形式是：

$$
\theta \leftarrow \theta - \eta \nabla_{\theta}\mathcal{L}
$$

其中 $\theta$ 表示网络中的所有参数，$\eta$ 表示学习率，$\nabla_{\theta}\mathcal{L}$ 表示损失函数对参数的梯度。

## 9. 小实验：用 PyTorch 训练 MLP 解决 XOR

下面我们用一个很小的 MLP 学习 XOR。这个实验的重点不是追求复杂代码，而是观察：加入隐藏层之后，模型可以学会单层感知机学不会的非线性关系。

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

X_tensor = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])

y_tensor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Tanh(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

loss_history = []

for epoch in range(2000):
    y_pred = model(X_tensor)
    loss = criterion(y_pred, y_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

with torch.no_grad():
    probs = model(X_tensor)
    preds = (probs >= 0.5).float()

print('预测概率:')
print(probs.numpy().round(4))
print('\n预测类别:')
print(preds.numpy().astype(int).ravel())
print('真实类别:')
print(y_tensor.numpy().astype(int).ravel())

plt.figure(figsize=(6, 3.6))
plt.plot(loss_history, color='#2563EB')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('MLP 学习 XOR 的损失曲线')
plt.grid(alpha=0.3)
plt.show()


## 10. 本节总结

这一节先记住四句话：

1. 感知机是用于分类的神经元，核心公式是 $\hat{y}=\operatorname{step}(\mathbf{w}^{T}\mathbf{x}+b)$。
2. 单层感知机的决策边界是线性的。
3. XOR 问题不是线性可分问题，所以单层感知机无法解决。
4. MLP 通过隐藏层和非线性激活函数，可以组合出更复杂的非线性关系。

下一节建议进入：前向传播与反向传播，重点理解 `loss.backward()` 到底在帮我们做什么。